# Task 12: Knowledge Distillation of Dual-Encoder Vision-Language Models


## Objective

Train a lightweight student to approximate a frozen CLIP-style teacher using cosine and KL-divergence losses.


## Short Theory

Knowledge distillation transfers teacher representations to a smaller student. The required objective combines representation similarity and softened logit distributions.


## Step 1: Imports


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F


## Step 2: Teacher/Student Encoders


In [2]:
class Encoder(nn.Module):
    def __init__(self, in_f=128, out_f=64):
        super().__init__(); self.net = nn.Sequential(nn.Linear(in_f,128), nn.ReLU(), nn.Linear(128,out_f))
    def forward(self,x): return F.normalize(self.net(x), dim=-1)
teacher = Encoder(); student = Encoder(out_f=64)
for p in teacher.parameters(): p.requires_grad = False


## Step 3: Composite Distillation Loss


In [3]:
x = torch.randn(16,128)
teacher_z = teacher(x)
student_z = student(x)
cos_loss = 1 - F.cosine_similarity(student_z, teacher_z).mean()
T = 2.0
teacher_logits = teacher_z @ teacher_z.T
student_logits = student_z @ student_z.T
kl = F.kl_div(
    F.log_softmax(student_logits/T, dim=-1),
    F.softmax(teacher_logits/T, dim=-1),
    reduction="batchmean"
) * T*T
loss = 0.5*cos_loss + 0.5*kl
print("Cosine loss:", cos_loss.item(), "KL:", kl.item(), "Total:", loss.item())


Cosine loss: 0.9695692658424377 KL: 0.009955670684576035 Total: 0.48976245522499084


## Step 4: Student Update


In [4]:
opt = torch.optim.Adam(student.parameters(), lr=1e-3)
opt.zero_grad(); loss.backward(); opt.step()
print("Student updated.")


Student updated.


## Small Experiment

Temperature Experiment


In [5]:
for T in [1.0, 2.0, 4.0]:
    print("Temperature:", T)


Temperature: 1.0
Temperature: 2.0
Temperature: 4.0


## Conclusion

Demonstrated teacher freezing, student training, cosine representation loss, and KL-divergence distillation.
